# OpenCLIP Transformer Multi-class Classification

## Import Libraries

In [1]:
import torch
from PIL import Image
import open_clip
import pandas as pd
import json

import numpy as np
import matplotlib.pyplot as plt
import os
from loguru import logger
import matplotlib.image as mpimg
from nazi_symbols_classification.training.data_preparation import get_image_paths
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, roc_curve, auc
import joblib 

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Data Preparation

Load OpenCLIP Model, Convert Images to Features Vectors and Store them in CSV Files

In [2]:
dir_name = os.path.dirname(os.getcwd())

'/Users/zhiweizhang/Projects/nazi_symbols_classification'

In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [3]:
def load_labels_df(dataset_path, sub_dataset_name):
    image_paths = get_image_paths(dataset_path, sub_folders=(sub_dataset_name,))
    logger.info(f"Number of images in {sub_dataset_name} is {len(image_paths)}")
    training_image_paths = [image.removeprefix(f"os.path.join(dataset_path, sub_dataset_name)/") for image in image_paths]
    training_labels = [os.path.basename(os.path.dirname(image)) for image in training_image_paths]
    labels = pd.DataFrame(dict(path=training_image_paths, nazi_cls=training_labels))
    return labels

In [3]:
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-classification", 
                         ("train", "test", "valid"))

In [39]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/train') and "non-nazi" not in image]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/test') and "non-nazi" not in image]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/valid') and "non-nazi" not in image]

In [40]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [3]:
def load_image(image_path):
    with torch.no_grad(), torch.cpu.amp.autocast():
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [44]:
with open("training_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(train_images), 200):
    data = preprocess_images(train_images[i:i+200])
    with open("training_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [45]:
with open("training_data_multiclass_label.csv", "w") as f:
    json.dump(y_train, f)

In [46]:
with open("validation_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(valid_images), 200):
    data = preprocess_images(valid_images[i:i+200])
    with open("validation_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [47]:
with open("validation_data_multiclass_label.csv", "w") as f:
    json.dump(y_valid, f)

In [48]:
with open("test_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(test_images), 200):
    data = preprocess_images(test_images[i:i+200])
    with open("test_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [49]:
with open("test_data_multiclass_label.csv", "w") as f:
    json.dump(y_test, f)

## Load Data

In [2]:
training_data = pd.read_csv("training_data_multiclass.csv")
validation_data = pd.read_csv("validation_data_multiclass.csv")
test_data = pd.read_csv("test_data_multiclass.csv")

In [3]:
training_data.shape, test_data.shape

((9009, 512), (1944, 512))

In [4]:
with open("training_data_multiclass_label.csv", "r") as f:
    y_train = json.load(f)
with open("test_data_multiclass_label.csv", "r") as f:
    y_test = json.load(f)
with open("validation_data_multiclass_label.csv", "r") as f:
    y_valid = json.load(f)

## Train Models and Evaluate

In [5]:
predicted_result = dict()

def gather_result(classifier):
    # print("score on test: " + str(classifier.score(pd.concat([validation_data, test_data]), y_valid + y_test)))
    # outputs = classifier.predict(pd.concat([validation_data, test_data]))
    # print(classification_report(y_valid + y_test, outputs, digits=3))
    # print("accuracy score:", accuracy_score(y_valid + y_test, outputs))
    print("score on test: " + str(classifier.score(test_data, y_test)))
    outputs = classifier.predict(test_data)
    print(classification_report(y_test, outputs, digits=3))
    print("accuracy score:", accuracy_score(y_test, outputs))
    predicted_result[type(classifier).__name__] = dict(y_true=y_test, outputs=outputs)

In [6]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data, y_train)

print("score on test: " + str(lr.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8202681794739557
CPU times: user 25.5 s, sys: 2min 5s, total: 2min 31s
Wall time: 6.79 s


In [7]:
gather_result(lr)

score on test: 0.816358024691358
                          precision    recall  f1-score   support

               black_sun      0.856     0.766     0.809       124
british_union_of_fascist      1.000     0.333     0.500        12
        broken_sun_cross      0.000     0.000     0.000        16
          happy_merchant      0.970     0.970     0.970        33
                  hitler      0.823     0.854     0.838       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.719     0.728     0.723       158
                siegrune      0.797     0.693     0.742       261
                ss_skull      0.741     0.768     0.754       220
   sturmabteilung_emblem      1.000     0.500     0.667         8
                swastika      0.846     0.923     0.883       886
              wolfsangel      0.611     0.333     0.431        33

                accuracy                 

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [8]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data, y_train)

print("score on test: " + str(sgd.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8396080453842186
CPU times: user 818 ms, sys: 800 ms, total: 1.62 s
Wall time: 660 ms


In [9]:
gather_result(sgd)

score on test: 0.8395061728395061
                          precision    recall  f1-score   support

               black_sun      0.873     0.831     0.851       124
british_union_of_fascist      1.000     0.667     0.800        12
        broken_sun_cross      1.000     0.188     0.316        16
          happy_merchant      0.943     1.000     0.971        33
                  hitler      0.818     0.849     0.833       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      1.000     1.000     1.000         3
                neo-nazi      0.713     0.804     0.756       158
                siegrune      0.850     0.716     0.778       261
                ss_skull      0.741     0.782     0.761       220
   sturmabteilung_emblem      1.000     0.625     0.769         8
                swastika      0.880     0.923     0.901       886
              wolfsangel      0.800     0.485     0.604        33

                accuracy                

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [10]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data, y_train)

print("score on test: " + str(knn.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8597215059308922
CPU times: user 9.58 s, sys: 358 ms, total: 9.94 s
Wall time: 807 ms


In [11]:
gather_result(knn)

score on test: 0.8631687242798354
                          precision    recall  f1-score   support

               black_sun      0.817     0.863     0.839       124
british_union_of_fascist      0.889     0.667     0.762        12
        broken_sun_cross      0.889     0.500     0.640        16
          happy_merchant      1.000     0.970     0.985        33
                  hitler      0.797     0.914     0.851       185
           hitler_salute      0.200     0.200     0.200         5
              judenstern      1.000     1.000     1.000         3
                neo-nazi      0.845     0.829     0.837       158
                siegrune      0.782     0.854     0.817       261
                ss_skull      0.789     0.750     0.769       220
   sturmabteilung_emblem      1.000     0.625     0.769         8
                swastika      0.938     0.907     0.923       886
              wolfsangel      0.688     0.667     0.677        33

                accuracy                

In [12]:
%%time

# import the library
from sklearn.svm import SVC

# instantiate & fit
svm=SVC(C=3)
svm.fit(training_data, y_train)

print("score on test: " + str(svm.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8780299123259412
CPU times: user 7.17 s, sys: 0 ns, total: 7.17 s
Wall time: 7.19 s


In [13]:
gather_result(svm)

score on test: 0.8765432098765432
                          precision    recall  f1-score   support

               black_sun      0.862     0.855     0.858       124
british_union_of_fascist      1.000     0.833     0.909        12
        broken_sun_cross      1.000     0.500     0.667        16
          happy_merchant      1.000     1.000     1.000        33
                  hitler      0.867     0.881     0.874       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      1.000     1.000     1.000         3
                neo-nazi      0.859     0.848     0.854       158
                siegrune      0.789     0.889     0.836       261
                ss_skull      0.788     0.759     0.773       220
   sturmabteilung_emblem      1.000     0.625     0.769         8
                swastika      0.925     0.929     0.927       886
              wolfsangel      0.909     0.606     0.727        33

                accuracy                

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [14]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data, y_train)

print("score on test: " + str(clf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5678184631253224
CPU times: user 906 ms, sys: 0 ns, total: 906 ms
Wall time: 909 ms


In [15]:
gather_result(clf)

score on test: 0.573045267489712
                          precision    recall  f1-score   support

               black_sun      0.000     0.000     0.000       124
british_union_of_fascist      0.000     0.000     0.000        12
        broken_sun_cross      0.000     0.000     0.000        16
          happy_merchant      0.000     0.000     0.000        33
                  hitler      0.862     0.605     0.711       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.000     0.000     0.000       158
                siegrune      0.644     0.249     0.359       261
                ss_skull      0.851     0.364     0.510       220
   sturmabteilung_emblem      0.000     0.000     0.000         8
                swastika      0.529     0.967     0.684       886
              wolfsangel      0.000     0.000     0.000        33

                accuracy                 

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [16]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data, y_train)

print("score on test: " + str(bg.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5941206807632801
CPU times: user 4.11 s, sys: 0 ns, total: 4.11 s
Wall time: 4.12 s


In [17]:
gather_result(bg)

score on test: 0.5997942386831275
                          precision    recall  f1-score   support

               black_sun      0.000     0.000     0.000       124
british_union_of_fascist      0.000     0.000     0.000        12
        broken_sun_cross      0.000     0.000     0.000        16
          happy_merchant      0.000     0.000     0.000        33
                  hitler      0.842     0.692     0.760       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.000     0.000     0.000       158
                siegrune      0.461     0.498     0.479       261
                ss_skull      0.720     0.409     0.522       220
   sturmabteilung_emblem      0.000     0.000     0.000         8
                swastika      0.591     0.923     0.720       886
              wolfsangel      0.000     0.000     0.000        33

                accuracy                

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [18]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data, y_train)

print("score on test: " + str(adb.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.6629706034038164
CPU times: user 57.4 s, sys: 0 ns, total: 57.4 s
Wall time: 57.5 s


In [19]:
gather_result(adb)

score on test: 0.6682098765432098
                          precision    recall  f1-score   support

               black_sun      0.744     0.492     0.592       124
british_union_of_fascist      1.000     0.083     0.154        12
        broken_sun_cross      0.000     0.000     0.000        16
          happy_merchant      1.000     0.606     0.755        33
                  hitler      0.804     0.730     0.765       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.553     0.361     0.437       158
                siegrune      0.506     0.460     0.482       261
                ss_skull      0.641     0.577     0.608       220
   sturmabteilung_emblem      0.000     0.000     0.000         8
                swastika      0.686     0.875     0.769       886
              wolfsangel      0.600     0.091     0.158        33

                accuracy                

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [20]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data, y_train)

print("score on test: " + str(gbc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.832645693656524
CPU times: user 17min 45s, sys: 0 ns, total: 17min 45s
Wall time: 17min 47s


In [21]:
gather_result(gbc)

score on test: 0.8410493827160493
                          precision    recall  f1-score   support

               black_sun      0.885     0.806     0.844       124
british_union_of_fascist      0.667     0.500     0.571        12
        broken_sun_cross      0.556     0.312     0.400        16
          happy_merchant      0.939     0.939     0.939        33
                  hitler      0.845     0.886     0.865       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      1.000     1.000     1.000         3
                neo-nazi      0.819     0.747     0.781       158
                siegrune      0.804     0.770     0.787       261
                ss_skull      0.775     0.750     0.762       220
   sturmabteilung_emblem      0.714     0.625     0.667         8
                swastika      0.869     0.924     0.896       886
              wolfsangel      0.720     0.545     0.621        33

                accuracy                

In [22]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data, y_train)

print("score on test: " + str(rf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5703971119133574
CPU times: user 8.32 s, sys: 0 ns, total: 8.32 s
Wall time: 8.33 s


In [23]:
gather_result(rf)

score on test: 0.573559670781893
                          precision    recall  f1-score   support

               black_sun      0.000     0.000     0.000       124
british_union_of_fascist      0.000     0.000     0.000        12
        broken_sun_cross      0.000     0.000     0.000        16
          happy_merchant      0.000     0.000     0.000        33
                  hitler      0.865     0.692     0.769       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.000     0.000     0.000       158
                siegrune      0.933     0.107     0.192       261
                ss_skull      0.879     0.427     0.575       220
   sturmabteilung_emblem      0.000     0.000     0.000         8
                swastika      0.521     0.976     0.680       886
              wolfsangel      0.000     0.000     0.000        33

                accuracy                 

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [24]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier

# 2) logistic regression =lr
lr=LogisticRegression(max_iter=5000)
# 3) random forest =rf
rf = RandomForestClassifier(n_estimators=30,max_depth=3)
# 4) suport vecotr mnachine = svm
svm=SVC(C=2, max_iter=5000)
evc=VotingClassifier(estimators=[('lr',lr),('rf',rf),('svm',svm)])
evc.fit(training_data, y_train)

print("score on test: " + str(evc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8378029912325942
CPU times: user 35.5 s, sys: 1min 43s, total: 2min 18s
Wall time: 13.6 s


In [25]:
gather_result(evc)

score on test: 0.8369341563786008
                          precision    recall  f1-score   support

               black_sun      0.847     0.847     0.847       124
british_union_of_fascist      1.000     0.417     0.588        12
        broken_sun_cross      1.000     0.250     0.400        16
          happy_merchant      1.000     0.970     0.985        33
                  hitler      0.863     0.849     0.856       185
           hitler_salute      0.000     0.000     0.000         5
              judenstern      0.000     0.000     0.000         3
                neo-nazi      0.755     0.741     0.748       158
                siegrune      0.827     0.732     0.776       261
                ss_skull      0.824     0.745     0.783       220
   sturmabteilung_emblem      1.000     0.500     0.667         8
                swastika      0.841     0.948     0.891       886
              wolfsangel      0.889     0.242     0.381        33

                accuracy                

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [26]:
joblib.dump(predicted_result, "openclip-encoder-multi-predicted-result.joblib")

['openclip-encoder-multi-predicted-result.joblib']